In [1]:
# !pip install pypulseq

In [1]:
import numpy as np 
import pypulseq as pq 
import matplotlib.pyplot as plt 

# maximum gradient amplitude [mT/m]
max_grad = 40.0

# maximum gradient slew rate [T/m/s]
# set super low here to eliminate PNS risks 
max_slew = 90.0

# duration of each sample in RF pulse [s]
rf_raster_time = 2.0e-6 

# necessary post-rf delay [s]
rf_ringdown_time = 60.0e-6

# necessary pre-rf delay [s]
rf_dead_time = 100.0e-6 

# necessary pre-adc delay [s] 
adc_dead_time = 40.0e-6 

# adc raster time [s]
adc_raster_time = 2.0e-6 

# gradient raster time [s]
grad_raster_time = 4.0e-6 

# total duration of each block must be evenly divisible by this number
block_duration_raster = 4.0e-6 

# delay inserted between segments for GE systems 
# added by GE pge2 interpreter 
end_of_segment_delay = 116.0e-6

# make the system object for PyPulseq 
system = pq.Opts(max_grad=max_grad,
                     grad_unit='mT/m',
                     max_slew=max_slew,
                     slew_unit='T/m/s',
                     rf_ringdown_time=rf_ringdown_time,
                     rf_dead_time=rf_dead_time,
                     rf_raster_time=rf_raster_time,
                     adc_dead_time=adc_dead_time,
                     adc_raster_time=adc_raster_time,
                     grad_raster_time=grad_raster_time,
                     block_duration_raster=block_duration_raster)

fovx = 0.22 # frequency encoding FOV [m]
fovy = 0.22 # phase encoding FOV [m]
dz = 0.005  # slice thickness [m]

# imaging matrix sizes 
nx = 128 
ny = 128 

# TODO: choose inversion times 
# T1 = [.1-2]s
TI = np.linspace(0.1, 2, 6, dtype=np.float32)
print(TI)

# TODO: choose this
TR = 3.0 # repetition time [s]

# TODO: choose this
flip_angle = 30.0 
alpha = flip_angle * np.pi / 180 

# TODO: choose this (make a multiple of 2 microseconds)
adc_dwell_time = 6.0e-6

# TODO: calculate frequency encoding gradient amplitude in units of Hz/m 
kx = 1/fovx
Gx_Hz_m = kx / adc_dwell_time 

# TODO: calculate duration of flat part of frequency encoding gradient 
Gx_flat_time = nx * adc_dwell_time

# make the frequency encoding gradient 
Gx = pq.make_trapezoid(channel='x', amplitude=Gx_Hz_m, flat_time=Gx_flat_time, system=system)

# Data collection object
adc = pq.make_adc(num_samples=nx, delay=Gx.rise_time, duration=Gx_flat_time, system=system)

# TODO: make the frequency encoding prephaser 
# Gx.area
area_Gx_pre = -0.5 * Gx.area # calculate this! 
Gx_pre = pq.make_trapezoid(channel='x', area=area_Gx_pre, system=system)

# TODO: calculate phase encoding areas (units of 1/m)
delta_ky = 1/fovy
phase_areas = np.arange(-ny/2, ny/2)*delta_ky # [m^-1] you make this, it should be a numpy array with length of ny
max_phase_area = np.max(np.abs(phase_areas))
phase_scales = phase_areas / max_phase_area # factor by which to scale amplitude of Gy gradient for each phase encoding step

# make the phase encoding gradient 
Gy = pq.make_trapezoid(channel='y', area=max_phase_area, system=system)

# TODO: make the inversion pulse 
inv_duration = 1.0e-3 
inv_flip_angle = np.pi # TODO: set this
rf_inv = pq.make_block_pulse(flip_angle=inv_flip_angle, duration=inv_duration, system=system)

# TODO: make the slice-selective excitation pulse 
exc_bw = 2000 # TODO: choose this
exc_duration = 0.003 # TODO: choose this 
exc_tbw = exc_bw * exc_duration 
rf_exc, gz, _ = pq.make_sinc_pulse(flip_angle=alpha, duration=exc_duration, time_bw_product=exc_tbw, slice_thickness=dz, return_gz=True, system=system)

area_Gz_pre = -0.5 * gz.area
Gz_pre = pq.make_trapezoid(channel='z', area=area_Gz_pre, system=system)

area_Gz_spoil = 4/dz
Gz_spoil = pq.make_trapezoid(channel='z', area=area_Gz_spoil, system=system)

# make the inversion time delays 
delay_TI = TI - 0.5*pq.calc_duration(rf_inv) - 0.5*pq.calc_duration(gz)
delay_TI = (np.round(delay_TI/grad_raster_time)).astype(np.int32)*grad_raster_time
TI_delay_object = []
for t in range(len(TI)):
    TI_delay_object.append(pq.make_delay(delay_TI[t]))

# make the repetition time delays 
delay_TR = TR - 0.5*pq.calc_duration(rf_inv) - TI - 0.5*pq.calc_duration(gz) 
delay_TR = (np.round(delay_TR/grad_raster_time)).astype(np.int32)*grad_raster_time
TR_delay_object = []
for t in range(len(TI)):
    TR_delay_object.append(pq.make_delay(delay_TR[t]))

[0.1  0.48 0.86 1.24 1.62 2.  ]


/var/folders/30/x3_16p4d5j5cq5v20dn6smqm0000gq/T/ipykernel_73114/2194559203.py:103: UserWarning: Specified RF delay 0.00 us is less than the dead time 100 us. Delay was increased to the dead time.
  rf_inv = pq.make_block_pulse(flip_angle=inv_flip_angle, duration=inv_duration, system=system)
/var/folders/30/x3_16p4d5j5cq5v20dn6smqm0000gq/T/ipykernel_73114/2194559203.py:109: UserWarning: Specified RF delay 0.00 us is less than the dead time 100 us. Delay was increased to the dead time.
  rf_exc, gz, _ = pq.make_sinc_pulse(flip_angle=alpha, duration=exc_duration, time_bw_product=exc_tbw, slice_thickness=dz, return_gz=True, system=system)


In [2]:
seq = pq.Sequence(system=system)

for inv in range(TI.size):

    for y in range(ny):

        seq.add_block(rf_inv, pq.make_label(label='TRID', type='SET', value=inv+1))

        seq.add_block(Gz_spoil)
        
        seq.add_block(TI_delay_object[inv])
    
        seq.add_block(rf_exc, gz)
    
        seq.add_block(Gz_pre, Gx_pre, pq.scale_grad(Gy, scale=phase_scales[y]))

        seq.add_block(Gx, adc)

        seq.add_block(Gz_spoil, pq.scale_grad(Gy, scale=-phase_scales[y]))

        seq.add_block(TR_delay_object[inv])

In [3]:
%matplotlib tk

In [4]:
seq.plot()

In [5]:

seqname = '/Users/nmickevicius/dev/MCW_BIOP_03-238_MRI/17_write_sequence/ir_gre.seq'

# write sequence file 
seq.write(seqname)

# convert to GE scanner format 
from adaptive_mri.psd import PSD
import tomllib 
locfile = '/Users/nmickevicius/dev/adaptive_mri/src/adaptive_mri/templates/local/mickevicius_m2_macbook_pro.toml'
sysfile = '/Users/nmickevicius/dev/adaptive_mri/src/adaptive_mri/templates/sys/mcw_cir_soref2.toml'
seqdir = '/Users/nmickevicius/dev/MCW_BIOP_03-238_MRI/17_write_sequence'

with open(locfile,'rb') as f: loc = tomllib.load(f) 
with open(sysfile,'rb') as f: amri_sys = tomllib.load(f)

psd_obj = PSD()
psd_obj.pulseq2toppe(loc, amri_sys, seqdir, seqname)

/Users/nmickevicius/dev/MCW_BIOP_03-238_MRI/.venv_amri/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
print(amri_sys)

{'vendor': 'GE', 'raw_data_patterns': ['*.h5'], 'remote_scanner_destination': 'sdc@141.106.208.26:/srv/nfs/psd/usr/psd/pulseq/v7', 'toppe_file_directory_on_scanner': '/srv/nfs/psd/usr/psd/pulseq/v7', 'remote_data_location': 'sdc@141.106.208.26:/usr/g/mrraw/adaptivemri', 'data_poll_interval': 1.0, 'data_size_stable_for': 3.0, 'max_b1': 24.0, 'max_grad': 40.0, 'max_slew': 90.0, 'rf_raster_time': 2e-06, 'rf_ringdown_time': 6e-05, 'rf_dead_time': 0.0001, 'adc_dead_time': 4e-05, 'adc_raster_time': 2e-06, 'grad_raster_time': 4e-06, 'block_duration_raster': 4e-06, 'end_of_segment_delay': 0.000116}


In [ ]:
# send sequence to scanner 
# psd_obj.seqsend(amri_sys, seqdir)

ssh_askpass: exec(/usr/X11R6/bin/ssh-askpass): No such file or directory
Host key verification failed.
scp: Connection closed
ssh_askpass: exec(/usr/X11R6/bin/ssh-askpass): No such file or directory
Host key verification failed.
scp: Connection closed
